# Laboratorio OpenDSS & py-dss-interface
## Módulo 0: Fundamentos de OpenDSS y Arquitectura de py-dss-interface

---

### Objetivos de Aprendizaje
Al finalizar este cuaderno, el estudiante será capaz de:
1. Comprender la arquitectura de **OpenDSS** (Open Distribution System Simulator) del EPRI y su modelo nodal en el dominio de la frecuencia.
2. Reconocer las ventajas del enlace directo mediante **`py-dss-interface`** (enlace en memoria / Direct C-API) frente a la tradicional interfaz COM en entornos Windows, Linux y macOS.
3. Instanciar el objeto principal del simulador, compilar un circuito elemental y verificar la convergencia del flujo de potencia.
4. Navegar la jerarquía de objetos: `Circuit`, `ActiveBus`, `ActiveElement`, `Solution` y `Lines`.
5. Extraer variables eléctricas básicas (tensiones de nodo en pu, corrientes de rama, pérdidas activas y reactivas).

---
### 1. Marco Teórico: Arquitectura de OpenDSS y py-dss-interface

OpenDSS resuelve las ecuaciones de red en el dominio de admitancias nodales:

$$\mathbf{I}_{	ext{iny}}(t, \mathbf{V}) = \mathbf{Y}_{	ext{bus}} \cdot \mathbf{V}(t)$$

Donde:
* $\mathbf{Y}_{	ext{bus}}$ es la matriz de admitancias nodales primitiva reducida ($N 	imes N$, simétrica o asimétrica en presencia de acoplamientos mutuos).
* $\mathbf{V}(t)$ es el vector complejo de tensiones nodales fase-tierra.
* $\mathbf{I}_{	ext{iny}}$ es el vector de corrientes inyectadas por fuentes, cargas no lineales, generadores y sistemas fotovoltaicos.

**¿Por qué `py-dss-interface`?**
Tradicionalmente, la automatización en Python dependía de la interfaz COM (`win32com.client`), la cual:
* Solo funciona en Microsoft Windows de 32/64 bits.
* Presenta una sobrecarga (*overhead*) significativa de serialización en simulaciones cuasi-estáticas de miles de pasos.
* Requiere registrar bibliotecas de tipos en el registro del sistema operativo.

`py-dss-interface`, desarrollado por Paulo Radatz, interactúa directamente con la biblioteca dinámica compilada en C/C++ (`OpenDSSDirect.dll` / `.so`), permitiendo acceso directo a la memoria a velocidades hasta 10 veces superiores y compatibilidad multiplataforma nativa.


In [1]:
# Importación de bibliotecas fundamentales
import py_dss_interface
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

print(f"py-dss-interface versión instalada: {py_dss_interface.__version__}")


py-dss-interface versión instalada: 2.3.0


---
### 2. Inicialización del Motor de Simulación y Compilación

Para interactuar con OpenDSS creamos una instancia de `py_dss_interface.DSS()`.

La instrucción principal para cargar un modelo es:
```python
dss.text(f"compile '{ruta_al_archivo_master.dss}'")
```

OpenDSS mantiene una cola de comandos accesibles mediante la propiedad `.text()`.


In [2]:
import sys, os
sys.path.append(r'C:\Users\gumor\Documents\OpenDSS-pyDSS-Lab')
from utils.lab_dss_toolkit import get_project_root
base_dir = get_project_root()
# Crear la instancia del motor OpenDSS
dss = py_dss_interface.DSS()
dss.dssinterface.allow_forms = False

# Definir la ruta del circuito de prueba (Red de Laboratorio BT)
# Resolver ruta raíz robusta
p = os.path.abspath(os.getcwd())
while p and os.path.basename(p) != 'OpenDSS-pyDSS-Lab':
    parent = os.path.dirname(p)
    if parent == p: break
    p = parent
circuit_path = os.path.join(p, 'data', 'circuit_lab_bt', 'Master_LabBT.dss')

print(f"Compilando circuito desde: {circuit_path}")
dss.text(f"compile '{circuit_path}'")

# Resolver el estado estacionario inicial (Snapshot Power Flow)
dss.solution.solve()

# Verificar estado de convergencia
convergio = bool(dss.solution.converged)
print(f"¿Flujo de potencia convergido?: {'[OK] Sí' if convergio else '[X] No'}")
print(f"Número de iteraciones empleadas: {dss.solution.iterations}")


Compilando circuito desde: C:\Users\gumor\Documents\OpenDSS-pyDSS-Lab\data\circuit_lab_bt\Master_LabBT.dss
¿Flujo de potencia convergido?: [OK] Sí
Número de iteraciones empleadas: 3


---
### 3. Navegación en la Jerarquía de Objetos

OpenDSS utiliza el concepto de **Elemento Activo** (*Active Element*) y **Barra Activa** (*Active Bus*).
Cuando se selecciona un elemento o barra, los métodos subsecuentes aplican sobre dicho objeto en memoria.

Veamos cómo explorar el circuito:
1. `dss.circuit.buses_names`: Lista de todas las barras de la red.
2. `dss.circuit.set_active_bus(nombre)`: Fija la barra activa.
3. `dss.bus.pu_voltages`: Devuelve fasores de tensión en por unidad $[V_{r1}, V_{i1}, V_{r2}, V_{i2}, \dots]$.
4. `dss.circuit.losses`: Devuelve las pérdidas totales de la red $[P_{	ext{loss}} 	ext{ (W)}, Q_{	ext{loss}} 	ext{ (var)}]$.


In [3]:
# Obtener información global del circuito
num_barras = dss.circuit.num_buses
num_elementos = dss.circuit.num_nodes
perdidas_kw = dss.circuit.losses[0] / 1000.0
perdidas_kvar = dss.circuit.losses[1] / 1000.0
potencia_total_kw = abs(dss.circuit.total_power[0])

print(f"Total de barras en la red: {num_barras}")
print(f"Total de nodos de fase: {num_elementos}")
print(f"Potencia total suministrada por la red: {potencia_total_kw:.2f} kW")
print(f"Pérdidas totales activas: {perdidas_kw:.3f} kW ({(perdidas_kw / potencia_total_kw * 100):.2f}% de la demanda)")
print(f"Pérdidas totales reactivas: {perdidas_kvar:.3f} kvar")


Total de barras en la red: 16
Total de nodos de fase: 32
Potencia total suministrada por la red: 64.49 kW
Pérdidas totales activas: 2.488 kW (3.86% de la demanda)
Pérdidas totales reactivas: 1.735 kvar


---
### 4. Inspección Nodal: Tensiones en Por Unidad (pu)

Recorramos cada barra para calcular la magnitud de tensión por fase:
$$V_{	ext{mag, pu}} = \sqrt{V_{	ext{real}}^2 + V_{	ext{imag}}^2}$$


In [4]:
# Extracción de magnitudes de tensión por barra
registros_barras = []

for b_name in dss.circuit.buses_names:
    dss.circuit.set_active_bus(b_name)
    nodos = dss.bus.nodes
    v_pu_raw = dss.bus.pu_voltages
    
    # Cada nodo tiene 2 valores consecutivos en v_pu_raw: [Real, Imag]
    tensiones = {}
    for idx, nodo in enumerate(nodos):
        r = v_pu_raw[idx * 2]
        im = v_pu_raw[idx * 2 + 1]
        tensiones[f"Fase_{nodo}"] = round(np.sqrt(r**2 + im**2), 4)
        
    registros_barras.append({
        "Barra": b_name,
        "Nodos": nodos,
        **tensiones
    })

df_tensiones = pd.DataFrame(registros_barras)
df_tensiones.head(10)


---
### 5. Ejercicio Guiado: Modificación Dinámica y Verificación

**Enunciado:**
1. Aumente la potencia activa de la carga `Load_8` (ubicada en `Node_6`, fase 2) de $11.0	ext{ kW}$ a $18.0	ext{ kW}$.
2. Vuelva a resolver el flujo de potencia (`dss.solution.solve()`).
3. Compare las pérdidas totales de la red y la caída de tensión en `LoadBus_8`.


In [5]:
# Estado base
p_perdidas_base = dss.circuit.losses[0] / 1000.0

# 1. Modificar potencia activa de Load_8
dss.loads.name = "load_8"
print(f"Potencia original de Load_8: {dss.loads.kw} kW")

dss.loads.kw = 18.0
print(f"Nueva potencia asignada a Load_8: {dss.loads.kw} kW")

# 2. Re-resolver flujo de potencia
dss.solution.solve()

# 3. Evaluar impacto
p_perdidas_nueva = dss.circuit.losses[0] / 1000.0
dss.circuit.set_active_bus("loadbus_8")
v_nueva_pu = np.sqrt(dss.bus.pu_voltages[0]**2 + dss.bus.pu_voltages[1]**2)

print(f"\n--- RESULTADOS COMPARATIVOS ---")
print(f"Pérdidas Base:  {p_perdidas_base:.3f} kW")
print(f"Pérdidas Nueva: {p_perdidas_nueva:.3f} kW (Incremento: +{(p_perdidas_nueva - p_perdidas_base):.3f} kW)")
print(f"Tensión resultante en LoadBus_8: {v_nueva_pu:.4f} pu")

# Restaurar valor original para no afectar ejercicios futuros
dss.loads.kw = 11.0
dss.solution.solve()
print("[OK] Carga Load_8 restaurada a 11.0 kW.")


Potencia original de Load_8: 11.0 kW
Nueva potencia asignada a Load_8: 18.0 kW

--- RESULTADOS COMPARATIVOS ---
Pérdidas Base:  2.488 kW
Pérdidas Nueva: 3.523 kW (Incremento: +1.035 kW)
Tensión resultante en LoadBus_8: 0.9176 pu
[OK] Carga Load_8 restaurada a 11.0 kW.


---
### Conclusiones del Módulo 0
1. `py-dss-interface` proporciona un control programático rápido, robusto y multiplataforma sobre OpenDSS.
2. Cada comando o asignación de propiedad (como `dss.loads.kw = 18.0`) modifica el estado en memoria de forma inmediata.
3. El cálculo de flujo de potencia mediante `dss.solution.solve()` actualiza las matrices y fasores de toda la red, permitiendo análisis en cadena y optimización.
